# Fitting a radial-velocity orbit with every parameter sampled

This is the **fully non-linear** companion of `fit_rv_orbit.ipynb`. That
notebook samples only the orbit *shape* `(P, e, tau)` and solves the three
amplitudes in closed form at each step -- three sampled dimensions, fast, and
immune to the basin trap. Here we sample **everything**: the shape, the two
angles and amplitudes `(omega, K, gamma)`, and a jitter term -- seven
parameters, no closed-form step anywhere. Same data, same conventions, and the
probability is written out in full so every choice is in front of you.

Why do this at all, if the linearised fit works? Two reasons.

1. **Posteriors for the amplitudes.** The linearised fit gives `K`, `omega`
   and `gamma` *conditionally* -- recovered per posterior draw of the shape.
   Sampling them directly gives them their own marginal posteriors, with the
   correlations between shape and amplitude visible in the corner plot.
2. **Jitter.** A per-epoch noise term beyond the quoted errors is not linear
   in anything; the only honest way to fit it is to sample it.

And one reason **not** to start here: seven dimensions from a cold start is
exactly where a sampler gets stuck. So we **seed** from the linearised
quick-look -- the same recipe `fit_rv_orbit.ipynb` builds -- and let the full
sampler *refine*. **A seed is initialisation, never a prior**: it decides
where the walkers start and nothing else.

We call only `orblet` atoms: `rv_curve` for the model, the prior classes,
`run_emcee_chains` for reproducible independent chains, `rhat_per_param` for
convergence, and the linear pieces for the seed. `emcee` and `corner` are
optional extras (`pip install "orblet[sampling,plots]"`).

**Units and conventions** (as in the sibling notebooks):

- `P` in **days**; converted to Keplerian years at the model interface.
- `e` in `[0, 1)`; `tau` periastron phase in `[0, 1)`, `tp = tau * P + T_REF`.
- `omega` in **radians**, **primary** frame: `v = gamma + K [cos(nu + omega) + e cos(omega)]`.
- `K`, `gamma`, `jitter` in **km/s**; the noise model is
  `sigma_eff^2 = sigma^2 + jitter^2`, epochs independent.
- RV alone fixes `sin i = 1`: `K` measures the mass function, not the companion mass.

> **Which fit is this?** The **fully non-linear** level: every parameter
> sampled, nothing solved in closed form. It is the third rung of the ladder
> whose first is `fit_rv_orbit.ipynb` (shape sampled, amplitudes solved). Start
> it from the first rung's answer; started cold, it can wander for a long time
> before finding the same basin the linearised fit finds in seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from astropy.timeseries import LombScargle

from orblet.simulate.bundles import load_simulated_inputs
from orblet.constants import MJD_J2010_TCB
from orblet.model import rv_curve
from orblet.priors import UniformPrior, LogUniformPrior
from orblet.sampling import run_emcee_chains, rhat_per_param
from orblet.chain_stats import chain_quantiles, chain_circular_summary
from orblet import rv_design_matrix, linear_solve_rv, recover_K, recover_omega

rng = np.random.default_rng(0)
DAYS_PER_YEAR = 365.25  # Keplerian year, matches the model

## 1. The data

The same synthetic SB1 orbit as `fit_rv_orbit.ipynb` (`seed=0`), on the same
time axis: days from J2010.0, with the reference epoch at zero.

In [ ]:
bundle = load_simulated_inputs(seed=0)
rv = bundle.rv_data
truth = bundle.truth

valid = np.asarray(rv['rv_validity_flag'])
t = np.asarray(rv['obs_time_rv'], dtype=float)[valid]          # days from J2010.0
v = np.asarray(rv['radial_velocity'], dtype=float)[valid]       # km/s
verr = np.asarray(rv['radial_velocity_err'], dtype=float)[valid]  # km/s
T_REF = 0.0

# The injected truth, on this notebook's time axis, for the comparison at the end.
tau_true = ((truth.tp_mjd - MJD_J2010_TCB - T_REF) / truth.P_days) % 1.0
theta_true = np.array([truth.P_days, truth.e, tau_true, truth.omega_rad % (2 * np.pi),
                       truth.K1_kms, truth.gamma_kms, 0.0])
print(f'{t.size} RV epochs over {t.min():.0f}..{t.max():.0f} days')
print('truth: P=%.2f d  e=%.3f  tau=%.3f  omega=%.3f rad  K=%.2f km/s  gamma=%.2f km/s  (no jitter injected)'
      % tuple(theta_true[:6]))

## 2. The seed: the linearised quick-look, in twenty lines

Exactly the recipe of `fit_rv_orbit.ipynb` §2-§4, compressed: a Lomb-Scargle
period, a Nelder-Mead search over the shape on the *marginal* likelihood (the
amplitudes integrated out analytically by `linear_solve_rv`), and the
closed-form amplitudes at the best shape. This point starts the walkers. It
carries no weight in the posterior.

In [ ]:
def model_at(theta, times):
    P, e, tau, omega, K, gamma = theta[:6]
    return rv_curve(times, period_yr=P / DAYS_PER_YEAR, ecc=e, omega_rad=omega,
                    tau=tau, K_kms=K, offset_kms=gamma, epoch_ref_mjd=T_REF)


def neg_marginal_logL(shape):
    P, e, tau = shape
    if not (10.0 < P < 500.0 and 0.0 <= e < 0.95 and 0.0 <= tau < 1.0):
        return 1e12
    X = rv_design_matrix(t, period_yr=P / DAYS_PER_YEAR, ecc=e, tau=tau, epoch_ref_mjd=T_REF)
    try:
        return -linear_solve_rv(v, verr, X).logL_marginal
    except Exception:
        return 1e12


ls = LombScargle(t, v, verr)
freq, power = ls.autopower(minimum_frequency=1 / 500.0, maximum_frequency=1 / 10.0, samples_per_peak=20)
pbest = float(1.0 / freq[np.argmax(power)])

best = None
for e0 in (0.1, 0.45, 0.7):
    for tau0 in np.linspace(0.05, 0.95, 5):
        res = minimize(neg_marginal_logL, [pbest, e0, tau0], method='Nelder-Mead')
        if best is None or res.fun < best.fun:
            best = res
P_s, e_s, tau_s = best.x
X = rv_design_matrix(t, period_yr=P_s / DAYS_PER_YEAR, ecc=e_s, tau=tau_s, epoch_ref_mjd=T_REF)
sol = linear_solve_rv(v, verr, X)
seed = np.array([P_s, e_s, tau_s, recover_omega(sol.beta) % (2 * np.pi), recover_K(sol.beta),
                 float(sol.beta[0]), 0.5])   # a small non-zero jitter to start from
print('seed (P, e, tau, omega, K, gamma, jitter):', np.round(seed, 3))

## 3. The probability, written out

Seven parameters, `theta = (P, e, tau, omega, K, gamma, jitter)`. The prior is
a product of orblet's prior classes -- each carries its own normalisation and
returns `-inf` outside its support -- and the likelihood is the Gaussian with
the jitter added in quadrature, the same noise model `orblet.likelihood`
implements. Nothing is hidden: change a prior by changing one line.

Two choices worth noticing. `P` and `K` get **log-uniform** priors: they are
scale parameters and a flat prior on a scale is a prior in favour of large
values. `omega` gets a flat prior on the full circle, and its posterior
summary below uses the *circular* mean and spread, because an angle near
`0` and one near `2 pi` are neighbours.

In [ ]:
PRIORS = [
    LogUniformPrior(10.0, 500.0),       # P (days)
    UniformPrior(0.0, 0.95),            # e
    UniformPrior(0.0, 1.0),             # tau
    UniformPrior(0.0, 2.0 * np.pi),     # omega (rad)
    LogUniformPrior(0.5, 200.0),        # K (km/s)
    UniformPrior(-300.0, 300.0),        # gamma (km/s)
    UniformPrior(0.0, 5.0),             # jitter (km/s)
]
LABELS = ['P (d)', 'e', 'tau', 'omega (rad)', 'K (km/s)', 'gamma (km/s)', 'jitter (km/s)']


def log_prior(theta):
    return float(sum(p.logpdf(x) for p, x in zip(PRIORS, theta)))


def log_likelihood(theta):
    jitter = theta[6]
    var = verr**2 + jitter**2
    r = v - model_at(theta, t)
    return float(-0.5 * np.sum(r**2 / var + np.log(2.0 * np.pi * var)))


def log_post(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_likelihood(theta)
    return lp + ll if np.isfinite(ll) else -np.inf


print('log posterior at the seed:', round(log_post(seed), 2))
print('log posterior at the truth (jitter -> 0.01):', round(log_post(np.r_[theta_true[:6], 0.01]), 2))

## 4. Sample: four independent chains

`run_emcee_chains` runs `n_chains` *independent* emcee ensembles from one
master seed -- one seed per chain, emcee's random state pinned per chain -- so
the whole run reproduces exactly and the chains can be compared to each other
for convergence. Each walker starts from `draw_init_fn`: the seed plus a small
scatter, **clipped into the support**. That clipping matters: a walker that
starts outside the prior gets `-inf` and emcee never moves it, which looks
like a bad posterior rather than a bad start.

In [ ]:
N_CHAINS, N_WALKERS, N_DIM, N_ITER = 4, 32, 7, 2000
SCATTER = np.array([0.5, 0.02, 0.01, 0.05, 1.0, 0.5, 0.1])   # start-cloud width per parameter


def draw_init(rng_):
    x = seed + SCATTER * rng_.standard_normal(N_DIM)
    x[1] = np.clip(x[1], 0.0, 0.94)          # e
    x[2] = x[2] % 1.0                        # tau
    x[3] = x[3] % (2.0 * np.pi)              # omega
    x[4] = max(x[4], 0.6)                    # K > 0
    x[6] = abs(x[6])                         # jitter >= 0
    return x


run = run_emcee_chains(log_post, n_chains=N_CHAINS, n_walkers=N_WALKERS, n_dim=N_DIM,
                       iterations=N_ITER, seed=42, draw_init_fn=draw_init, engine_name='rv-all')
print('acceptance per chain:', np.round(run.per_chain_acceptance, 3))
print('chain seeds:', run.chain_seeds)

## 5. Did it converge?

Discard the first half as warm-up, then ask two questions. Do the four chains
agree with each other (**R-hat**, close to 1 when they do)? And within each
chain, has it stopped drifting (R-hat on the two *halves* of every chain, so
eight "chains" -- a drifting chain disagrees with itself)?

In [ ]:
BURN = N_ITER // 2
per_chain = [s[BURN:].reshape(-1, N_DIM) for s in run.per_chain_samples]   # (draws, n_dim) each
stacked = np.stack(per_chain)                                                # (n_chains, draws, n_dim)
half = stacked.shape[1] // 2
split = np.concatenate([stacked[:, :half], stacked[:, half:2 * half]])       # (2 n_chains, half, n_dim)

for name, rh in zip(LABELS, rhat_per_param(split)):
    print(f'  split R-hat  {name:14s} {rh:.3f}')
flat = stacked.reshape(-1, N_DIM)
print(f'\n{flat.shape[0]} posterior draws after warm-up')

## 6. The answer, against the truth

Median and 16-84% interval per parameter, with `omega` summarised on the
circle. Read the jitter row carefully: the injected orbit had **no** jitter,
yet the posterior median is not zero. With 36 epochs the data barely
constrain a scale of a few tenths of a km/s, and a *flat* prior on a weakly
constrained scale puts most of its mass away from zero -- the median is a
fact about the prior, not a detection. The honest statement is the upper
bound; a log-uniform prior would put the median near the floor instead.
Either way the seven-parameter fit reproduces the six the linearised fit
found, which is the point.

In [ ]:
print(f'{"parameter":14s} {"truth":>9s} {"median":>9s} {"16%":>9s} {"84%":>9s}')
for j, name in enumerate(LABELS):
    if j == 3:
        c = chain_circular_summary(flat[:, j])
        print(f'{name:14s} {theta_true[j]:9.3f} {c["circmean_rad"]:9.3f}   circular std {c["circstd_rad"]:.3f}')
        continue
    q = chain_quantiles(flat[:, j], levels=(0.16, 0.5, 0.84))
    print(f'{name:14s} {theta_true[j]:9.3f} {q["q500"]:9.3f} {q["q160"]:9.3f} {q["q840"]:9.3f}')

In [ ]:
theta_med = np.median(flat, axis=0)
theta_med[3] = chain_circular_summary(flat[:, 3])['circmean_rad']
P_m, tau_m = theta_med[0], theta_med[2]
phase = ((t - (tau_m * P_m + T_REF)) / P_m) % 1.0
ph_grid = np.linspace(0.0, 1.0, 400)
t_grid = ph_grid * P_m + tau_m * P_m + T_REF

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
for j in rng.choice(flat.shape[0], size=100, replace=False):
    ax1.plot(ph_grid, model_at(flat[j], t_grid), color='C0', alpha=0.05, lw=1)
ax1.errorbar(phase, v, yerr=verr, fmt='o', color='k', ms=4, capsize=2, zorder=5)
ax1.set_ylabel('radial velocity (km/s)'); ax1.set_title('phase-folded RV, 100 posterior curves')
resid = v - model_at(theta_med, t)
ax2.axhline(0.0, color='C3', lw=1)
ax2.errorbar(phase, resid, yerr=verr, fmt='o', color='k', ms=4, capsize=2)
ax2.set_xlabel('orbital phase'); ax2.set_ylabel('O - C (km/s)')
plt.show()

## 7. The corner plot (needs the `plots` extra)

The reason to sample everything: the shape-amplitude correlations are now
*measured*, not assumed away. Look for the `e`-`omega` and `K`-`e` shapes.

In [ ]:
try:
    import corner
except ImportError:
    print('corner is not installed; `pip install "orblet[plots]"` to draw the corner plot')
else:
    fig = corner.corner(flat, labels=LABELS, truths=theta_true, show_titles=True, title_fmt='.3f')
    plt.show()

## Summary

Every parameter of the SB1 orbit sampled directly -- shape, angles, amplitudes
and a jitter -- from `orblet`'s atoms, with the probability written out and
four independent chains checked against each other. The answer agrees with
the linearised fit of `fit_rv_orbit.ipynb`, as it must: they are the same
likelihood. What this route adds is the amplitudes' own posteriors and the
jitter; what it costs is seven dimensions instead of three, which is why it
starts from the linearised answer rather than replacing it.

The same pattern -- linearised seed, full-parameter refinement with
`run_emcee_chains` -- carries to the astrometric and joint fits. For the joint
case there is an intermediate rung first: share the **angles** between the two
channels, not just the shape.